# What this simulation is doing

This notebook runs **2D incompressible hydrodynamics** (`Hydrodynamics2D`).

- **State variables**: `u` (x-velocity), `v` (y-velocity), `p` (pressure).
- **Dynamics**: advection + viscosity + forcing, then pressure projection to enforce incompressibility (`∇·v = 0`).

## Initial Conditions

- Velocity starts from a divergence-free random perturbation field (or smooth seeded flow, depending on settings).
- Pressure is initialized near zero and quickly adjusts through projection.

## Boundary Conditions

- **Periodic in x and y**: flow exiting one edge re-enters on the opposite edge.
- No wall/no-slip boundaries are present in this setup.

## What to expect

- Early forcing creates large coherent structures.
- Shear instabilities bend and roll these structures into vortices over time.

## Quick parameter guide

- `n`: `64` (fast) to `128` (sharper structures)
- `T`: `0.8`–`2.0` (longer to see instability growth)
- `cfl`: `0.25`–`0.35` (lower is more robust)
- `nu`: `5e-4`–`3e-3` (lower gives richer turbulence-like motion)
- `force`: `0.1`–`0.4` (higher drives stronger dynamics)

In [ ]:
from autosim.simulations import Hydrodynamics2D as Sim

# Kolmogorov Flow settings (unstable shear flow)
sim = Sim(
    return_timeseries=True,
    log_level="warning",
    n=64,
    T=3.0,       # Longer duration to observe instability
    dt=0.01,
    cfl=0.30,
    parameters_range={
        "nu": (1e-3, 2e-3),   # Viscosity
        # TODO: try lower viscosity to see more turbulent behavior
        # "nu": (3e-4, 2e-4),   # Viscosity
        # "nu": (1e-4, 2e-4),   # Viscosity
        "force": (1.0, 3.0),  # Forcing strength
    },
)
data = sim.forward_samples_spatiotemporal(n=1, random_seed=42)

In [ ]:
# Check shape
data["data"].shape

In [ ]:
from autosim.utils import plot_spatiotemporal_video

# View all components (u, v, p) to understand the dynamics
# u: Horizontal velocity (driven by forcing)
# v: Vertical velocity (generated by instability)
# p: Pressure (enforces incompressibility)
anim = plot_spatiotemporal_video(
    data["data"],
    batch_idx=1, # Pick a sample
    channel_names=["u", "v", "p"],
)

In [ ]:
from IPython.display import HTML

HTML(anim.to_jshtml())